In [ ]:
from openai import OpenAI
import os
import sys

api_key = os.environ.get("NVIDIA_API_KEY", "")

_USE_COLOR = sys.stdout.isatty() and os.getenv("NO_COLOR") is None
_REASONING_COLOR = "\033[90m" if _USE_COLOR else ""
_RESET_COLOR = "\033[0m" if _USE_COLOR else ""

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = api_key
)


completion = client.chat.completions.create(
  model="z-ai/glm-5.1",
  messages=[{"role":"user","content":"Which number is larger, 9.11 or 9.8?"}],
  temperature=1,
  top_p=1,
  max_tokens=8192,
  extra_body={"chat_template_kwargs":{"enable_thinking":True,"clear_thinking":False}},
  stream=True
)

for chunk in completion:
  if not getattr(chunk, "choices", None):
    continue
  if len(chunk.choices) == 0 or getattr(chunk.choices[0], "delta", None) is None:
    continue
  delta = chunk.choices[0].delta
  reasoning = getattr(delta, "reasoning_content", None)
  if reasoning:
    print(f"{_REASONING_COLOR}{reasoning}{_RESET_COLOR}", end="")
  if getattr(delta, "content", None) is not None:
    print(delta.content, end="")

In [ ]:
import os
import pandas as pd
from openai import OpenAI

# Setup client
api_key = os.environ.get("NVIDIA_API_KEY", "")
print(f"Using API Key: {api_key}")
client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = api_key
)

# Load dataset and get one sample
input_csv = "src/train.csv"
df = pd.read_csv(input_csv)
sample_row = df.iloc[0]

prompt_text = sample_row['prompt']
correct_answer = str(sample_row['answer']).strip()

print(f"--- PROMPT ---\n{prompt_text}\n")
print(f"--- EXPECTED ANSWER ---\n{correct_answer}\n")
print("--- MODEL OUTPUT ---")

system_prompt = "You are a helpful assistant. Please think step-by-step to solve the problem and provide your reasoning in English."
user_prompt = f"{prompt_text}\n\nPlease provide your final answer clearly."

completion = client.chat.completions.create(
  model="z-ai/glm-5.1",
  messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ],
  temperature=0.75,
  top_p=0.95,
  max_tokens=8192,
  extra_body={"chat_template_kwargs":{"enable_thinking":True,"clear_thinking":False}},
  stream=True
)

for chunk in completion:
    if not getattr(chunk, "choices", None):
        continue
    delta = chunk.choices[0].delta
    
    # Print reasoning
    r_text = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
    if r_text:
        print(r_text, end="")
        
    # Print content
    c_text = delta.content
    if c_text is not None:
        print(c_text, end="")
        
print("\n\n--- DONE ---")

## MISTRAL LARGE-3


In [ ]:

import requests, base64

invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
stream = True


headers = {
  "Authorization": f"Bearer {os.environ.get('NVIDIA_API_KEY', '')}",
  "Accept": "text/event-stream" if stream else "application/json"
}

payload = {
  "model": "mistralai/mistral-large-3-675b-instruct-2512",
  "messages": [{"role":"user","content":"Hey??"}],
  "max_tokens": 2048,
  "temperature": 0.15,
  "top_p": 1.00,
  "frequency_penalty": 0.00,
  "presence_penalty": 0.00,
  "stream": stream
}

response = requests.post(invoke_url, headers=headers, json=payload)

if stream:
    for line in response.iter_lines():
        if line:
            print(line.decode("utf-8"))
else:
    print(response.json())


## DEEPSEEK V4-Pro


In [ ]:
import os
import pandas as pd
from openai import OpenAI, APITimeoutError, APIConnectionError, APIStatusError
from tqdm import tqdm
import time

# ------------------------------------------------------------------
# 1. API setup with explicit timeout (60 seconds)
# ------------------------------------------------------------------
api_key = os.environ.get("NVIDIA_API_KEY", "")
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
    timeout=60.0   # adjust if you see frequent timeouts
)

# ------------------------------------------------------------------
# 2. Load dataset & resume support (instant save after each row)
# ------------------------------------------------------------------
input_csv = "src/train.csv"
output_csv = "src/train_with_cot.csv"
df = pd.read_csv(input_csv)

if os.path.exists(output_csv):
    processed_df = pd.read_csv(output_csv)
    processed_ids = set(processed_df['id'].astype(str))
    print(f"Resuming from {len(processed_ids)} already processed rows.")
else:
    processed_ids = set()
    # Create the output file with headers
    pd.DataFrame(columns=['id', 'prompt', 'answer', 'generated_cot']).to_csv(output_csv, index=False)
    print("Starting fresh output file.")

# ------------------------------------------------------------------
# 3. System prompt: concise reasoning, correct answer already given
# ------------------------------------------------------------------
SYSTEM_PROMPT = (
    "You are a helpful assistant. Given a problem and its correct answer, "
    "produce a detailed but CONCISE step-by-step reasoning in English that leads to that answer. "
    "Keep your reasoning efficient: focus only on the key deductions, avoid dead ends, "
    "do not over-explain trivial steps. The output space is limited, so every sentence must count. "
    "End your final response with the answer clearly stated."
)

# ------------------------------------------------------------------
# 4. Parameters to avoid exceeding token limit
# ------------------------------------------------------------------
MODEL = "deepseek-ai/deepseek-v4-pro"
MAX_TOKENS = 8192          # Increase to 16384 if your plan allows it
TEMPERATURE = 0.5          # Lower = more focused, less wandering
TOP_P = 0.95
REASONING_EFFORT = "max"  # "low" if you still hit the token cap

MAX_RETRIES = 3
RETRY_BASE_DELAY = 5.0     # seconds – increased for timeout/connection errors
RATE_LIMIT_DELAY = 1.5     # seconds between requests for 40 RPM

# ------------------------------------------------------------------
# 5. Process loop
# ------------------------------------------------------------------
for _, row in tqdm(df.iterrows(), total=len(df), desc="Generating CoT"):
    row_id = str(row['id'])
    if row_id in processed_ids:
        continue

    prompt_text = row['prompt']
    correct_answer = str(row['answer']).strip()

    # Build user prompt that includes the correct answer
    user_prompt = (
        f"Problem: {prompt_text}\n"
        f"Correct answer: {correct_answer}\n\n"
        "Write a short but complete reasoning that arrives at this answer. "
        "Be succinct - the output space is limited. Finish with the answer."
    )

    success = False
    retries_left = MAX_RETRIES
    while retries_left > 0:
        try:
            # ---------------- API call ----------------
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_TOKENS,
                extra_body={
                    "chat_template_kwargs": {
                        "thinking": True,
                        "reasoning_effort": REASONING_EFFORT
                    }
                },
                stream=True
            )

            reasoning_pieces = []
            content_pieces = []

            for chunk in completion:
                if not getattr(chunk, "choices", None):
                    continue
                delta = chunk.choices[0].delta
                # Collect reasoning (thinking block)
                r_text = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
                if r_text:
                    reasoning_pieces.append(r_text)
                if delta.content:
                    content_pieces.append(delta.content)

            full_reasoning = "".join(reasoning_pieces)
            full_content = "".join(content_pieces)

            # ---------------- Validation ----------------
            if full_reasoning.strip() and correct_answer in full_content:
                # Save immediately – appends to CSV
                new_row = pd.DataFrame([{
                    'id': row['id'],
                    'prompt': row['prompt'],
                    'answer': row['answer'],
                    'generated_cot': full_reasoning
                }])
                new_row.to_csv(output_csv, mode='a', header=False, index=False)
                processed_ids.add(row_id)
                success = True
            else:
                tqdm.write(f"⚠️  Empty CoT or missing answer for id {row_id} – skipping.")

            break  # exit retry loop (success or non‑retryable skip)

        # ---------------- Error handling ----------------
        except (APITimeoutError, APIConnectionError) as e:
            retries_left -= 1
            if retries_left > 0:
                delay = RETRY_BASE_DELAY * (2 ** (MAX_RETRIES - retries_left))
                tqdm.write(f"⏳ Timeout/connection error for id {row_id}: {e}. Retrying in {delay:.1f}s...")
                time.sleep(delay)
            else:
                tqdm.write(f"❌ Failed after {MAX_RETRIES} retries for id {row_id}: {e}")

        except APIStatusError as e:
            retries_left -= 1
            if retries_left > 0:
                delay = 10.0   # longer pause for rate limits / server errors
                tqdm.write(f"⏳ API error for id {row_id}: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            else:
                tqdm.write(f"❌ Failed after {MAX_RETRIES} retries for id {row_id}: {e}")

        except Exception as e:
            retries_left -= 1
            if retries_left > 0:
                tqdm.write(f"⚠️  Unexpected error for id {row_id}: {e}. Retrying...")
                time.sleep(RETRY_BASE_DELAY)
            else:
                tqdm.write(f"❌ Failed after {MAX_RETRIES} retries for id {row_id}: {e}")

    # ---------------- Rate-limit pacing ----------------
    if success:
        time.sleep(RATE_LIMIT_DELAY)
    else:
        # Even on failure, don't hammer the API
        time.sleep(2)

In [4]:
import os 
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = os.environ.get("NVIDIA_API_KEY", "")
)

print("Client initialized successfully.")

completion = client.chat.completions.create(
  model="deepseek-ai/deepseek-v4-pro",
  messages=[{"role":"user","content":"""
You are a helpful assistant. Given a problem and its correct answer,
produce a detailed but CONCISE step-by-step reasoning in English that leads to that answer.
Keep your reasoning efficient: focus only on the key deductions, avoid dead ends,
do not over-explain trivial steps. The output space is limited, so every sentence must count.
End your final response with the answer clearly stated.

In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. 
The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100 and the correct answer is 10010111"""}],
  temperature=1,
  top_p=0.95,
  max_tokens=8192,
  extra_body={"chat_template_kwargs":{"thinking":True,"reasoning_effort":"max"}},
  stream=True
)

for chunk in completion:
  if not getattr(chunk, "choices", None):
    continue
  reasoning = getattr(chunk.choices[0].delta, "reasoning", None) or getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices and chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

Client initialized successfully.
We are given several examples of an 8-bit binary transformation. We need to find the rule and apply it to 00110100 to get 10010111.

Let's list the inputs and outputs:

1. 01010001 -> 11011101
2. 00001001 -> 01101101
3. 00010101 -> 01010101
4. 11111111 -> 10000001
5. 10011101 -> 01000101
6. 00111011 -> 00001001
7. 10111101 -> 00000101
8. 00100110 -> 10110011

We need to find a transformation involving bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions. It might be a specific algorithm like a hash or a known bit manipulation. Let's analyze the bits.

Write them as bits, maybe we can find a pattern. Let's denote bits as b7 b6 b5 b4 b3 b2 b1 b0 (leftmost is MSB). Input and output are 8 bits.

Let's look for simple operations: maybe it's a linear feedback shift register (LFSR) or something. But we need to find a consistent rule.

Let's try to see if there's a relationship between input and output bits. Perhaps the output is 

RemoteProtocolError: peer closed connection without sending complete message body (incomplete chunked read)